# TuSimple Lane Segmentation without Google Drive
Upload `tusimple_colab_subset_1000.zip` when prompted.

In [ ]:
# Lane Segmentation TuSimple without Google Drive
# Upload local zip: tusimple_colab_subset_1000.zip
!pip install -q opencv-python matplotlib tqdm

from google.colab import files
uploaded = files.upload()

import os, json, zipfile
import cv2
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
from tensorflow.keras import layers, Model
from tqdm import tqdm

zip_name = next(iter(uploaded.keys()))
zipfile.ZipFile(zip_name).extractall('/content')

print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))

DATASET_PATH = '/content/TUSimple/train_set'
JSON_PATH = DATASET_PATH + '/label_data_0313.json'
IMG_HEIGHT = 256
IMG_WIDTH = 256
LIMIT = 1000
EPOCHS = 10
BATCH_SIZE = 8

items = [json.loads(line) for line in open(JSON_PATH) if line.strip()]


def create_mask(lanes, h_samples, image_height, image_width, line_width=5):
    mask = np.zeros((image_height, image_width), dtype=np.uint8)
    for lane in lanes:
        points = [(int(x), int(y)) for x, y in zip(lane, h_samples) if x != -2]
        if len(points) > 1:
            cv2.polylines(mask, [np.array(points)], False, 255, line_width)
    return mask

images, masks = [], []
for item in tqdm(items[:LIMIT], desc='Loading TuSimple'):
    image_path = os.path.join(DATASET_PATH, item['raw_file'])
    image = cv2.imread(image_path)
    if image is None:
        continue
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    mask = create_mask(item['lanes'], item['h_samples'], image.shape[0], image.shape[1])
    image = cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT))
    mask = cv2.resize(mask, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST)
    images.append(image.astype(np.float32) / 255.0)
    masks.append(mask.astype(np.float32) / 255.0)

X = np.array(images)
Y = np.array(masks)[..., np.newaxis]
print('Images:', X.shape)
print('Masks:', Y.shape)


def conv_block(x, filters):
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    x = layers.Conv2D(filters, 3, padding='same', activation='relu')(x)
    return x

inputs = layers.Input((IMG_HEIGHT, IMG_WIDTH, 3))
c1 = conv_block(inputs, 64)
p1 = layers.MaxPooling2D()(c1)
c2 = conv_block(p1, 128)
p2 = layers.MaxPooling2D()(c2)
c3 = conv_block(p2, 256)
u1 = layers.UpSampling2D()(c3)
u1 = layers.Concatenate()([u1, c2])
c4 = conv_block(u1, 128)
u2 = layers.UpSampling2D()(c4)
u2 = layers.Concatenate()([u2, c1])
c5 = conv_block(u2, 64)
outputs = layers.Conv2D(1, 1, activation='sigmoid')(c5)
model = Model(inputs, outputs)

SMOOTH = 1e-6

def dice_coef(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    return (2.0 * intersection + SMOOTH) / (tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) + SMOOTH)


def iou_coef(y_true, y_pred):
    y_true_f = tf.reshape(tf.cast(y_true, tf.float32), [-1])
    y_pred_f = tf.reshape(tf.cast(y_pred, tf.float32), [-1])
    intersection = tf.reduce_sum(y_true_f * y_pred_f)
    union = tf.reduce_sum(y_true_f) + tf.reduce_sum(y_pred_f) - intersection
    return (intersection + SMOOTH) / (union + SMOOTH)


def bce_dice_loss(y_true, y_pred):
    bce = tf.keras.losses.binary_crossentropy(y_true, y_pred)
    return bce + (1.0 - dice_coef(y_true, y_pred))

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=bce_dice_loss,
    metrics=['accuracy', dice_coef, iou_coef]
)

history = model.fit(X, Y, validation_split=0.2, epochs=EPOCHS, batch_size=BATCH_SIZE)

OUT_DIR = '/content/results'
os.makedirs(OUT_DIR, exist_ok=True)
model_path = OUT_DIR + '/unet_tusimple.keras'
model.save(model_path)

idx = min(5, len(X) - 1)
pred = model.predict(X[idx:idx+1])[0]

plt.figure(figsize=(12, 5))
plt.subplot(1, 3, 1)
plt.imshow(X[idx])
plt.title('Image')
plt.axis('off')
plt.subplot(1, 3, 2)
plt.imshow(Y[idx].squeeze(), cmap='gray')
plt.title('True mask')
plt.axis('off')
plt.subplot(1, 3, 3)
plt.imshow(pred.squeeze(), cmap='gray')
plt.title('Predicted mask')
plt.axis('off')
result_path = OUT_DIR + '/example_segmentation.png'
plt.savefig(result_path, dpi=160)
plt.show()

print('Saved model:', model_path)
print('Saved example:', result_path)
files.download(result_path)
files.download(model_path)
